# 04 · Track and intensity models + algorithm comparison
One table (`storm_fixes`), one feature set, many targets. Every algorithm gets the **same features and years**:
hyper-parameters are tuned by 5-fold cross-validation grouped by year **inside the training years (1990–2002)**;
scores below are on the untouched validation years **(2003–2004)**. Test years (2007–2008) are only used in notebook 06.

* **Track:** Δlat, Δlon at +6/12/18/24 h → error in km, skill vs CLIPER
* **Intensity:** wind change at +6/12/18/24 h → MAE in kt, skill vs SHIFOR
* **Rapid intensification** (+30 kt in 24 h): probability → Brier, ROC-AUC
* **Peak intensity:** lifetime maximum wind → MAE, IMD grade accuracy

**Runtime:** CPU (High-RAM not needed). **Takes:** ~20–40 min. Set `FAST = True` for a quick pass.

In [ ]:
# ▶ Setup: connect Google Drive, install missing packages, load the project code
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm", "xgboost"], check=False)

PROJECT_DIR = "/content/drive/MyDrive/CycloneProject"      # <- change only if you put the project elsewhere
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np, pandas as pd
from src import config as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("project:", C.PROJECT, "| ERA5:", C.ERA5_ROOT)

In [ ]:
import json, joblib, warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.tree import DecisionTreeRegressor
from src import baselines as BL, metrics as M, plots as P, models as MD
from src.common import load_table, imd_grade
from src.labels import TARGET_TRACK, TARGET_INT

FAST = False
df = load_table(C.out("tables") / "storm_fixes")
FEATS = MD.storm_feature_columns(df)
tr, va = df[df.split == "train"].copy(), df[df.split == "val"].copy()
print(f"train {len(tr)} rows / {tr.SID.nunique()} storms | val {len(va)} rows / {va.SID.nunique()} storms | {len(FEATS)} features")
json.dump(FEATS, open(C.out("models") / "storm_features.json", "w"))

## Baselines

In [ ]:
PRED = {}                                        # model name -> DataFrame of val predictions (all targets)
cl = BL.cliper(TARGET_TRACK).fit(tr); sh = BL.shifor(TARGET_INT).fit(tr)
PRED["Persistence"] = pd.concat([BL.track_persistence(va), BL.intensity_persistence(va)], axis=1)
PRED["CLIPER"] = cl.predict(va)
PRED["SHIFOR"] = sh.predict(va)

## Tune and train every algorithm
Parameters are tuned once per target family on the 24 h target, then reused for all leads.

In [ ]:
zoo = MD.regressor_zoo()
if FAST:
    zoo = {k: v for k, v in zoo.items() if k in ("Ridge (linear)", "Random Forest", "LightGBM", "XGBoost")}
FAMILIES = {"dlat": [t for t in TARGET_TRACK if "dlat" in t], "dlon": [t for t in TARGET_TRACK if "dlon" in t],
            "dwind": TARGET_INT}
BEST, CV, FITTED = {}, {}, {}
for name, (make, grid) in zoo.items():
    preds = {}
    for fam, targets in FAMILIES.items():
        rep = targets[-1]
        ok = tr[rep].notna().values
        params, cv = MD.tune(make, grid, tr.loc[ok, FEATS], tr.loc[ok, rep].values, tr.loc[ok, "year"].values)
        BEST[(name, fam)], CV[(name, fam)] = params, cv
        for t in targets:
            ok = tr[t].notna().values
            m = make(**params).fit(tr.loc[ok, FEATS], tr.loc[ok, t].values)
            FITTED[(name, t)] = m
            preds[t] = m.predict(va[FEATS])
    PRED[name] = pd.DataFrame(preds, index=va.index)
    print(f"{name:20s} done | CV MAE 24 h: dlat {CV[(name,'dlat')]:.3f}  dlon {CV[(name,'dlon')]:.3f}  dwind {CV[(name,'dwind')]:.2f}")

## Stacked ensemble
Top 3 models by cross-validation score (training years only) → out-of-fold predictions → ridge regression on top.

In [ ]:
stack = {}
for fam, targets in FAMILIES.items():
    top = sorted([n for n in zoo], key=lambda n: CV[(n, fam)])[:3]
    for t in targets:
        ok = tr[t].notna().values
        X, y, g = tr.loc[ok, FEATS], tr.loc[ok, t].values, tr.loc[ok, "year"].values
        oof = np.column_stack([MD.oof_predictions(zoo[n][0], BEST[(n, fam)], X, y, g) for n in top])
        meta = Ridge(alpha=1.0).fit(np.nan_to_num(oof), y)
        base_val = np.column_stack([FITTED[(n, t)].predict(va[FEATS]) for n in top])
        stack[t] = meta.predict(base_val)
        FITTED[("Stacked ensemble", t)] = (top, meta)
    print(fam, "stack of", top)
PRED["Stacked ensemble"] = pd.DataFrame(stack, index=va.index)

## Results: track and intensity

In [ ]:
rows = []
for name, pr in PRED.items():
    for L in C.LEADS_H:
        if f"y_dlat_{L}" in pr:
            ok = va[f"y_dlat_{L}"].notna().values
            e = M.track_error_km(va[ok], pr[ok], L)
            rows.append(dict(model=name, lead_h=L, target="track", track_error_km=np.nanmean(e), n=int(ok.sum())))
        if f"y_dwind_{L}" in pr:
            y = va[f"y_dwind_{L}"].values; p = np.asarray(pr[f"y_dwind_{L}"])
            rows.append(dict(model=name, lead_h=L, target="intensity", mae_kt=M.mae(y, p), rmse_kt=M.rmse(y, p),
                             n=int(np.isfinite(y).sum())))
res = pd.DataFrame(rows)
trk = res[res.target == "track"].copy(); itn = res[res.target == "intensity"].copy()
ref_t = trk[trk.model == "CLIPER"].set_index("lead_h")["track_error_km"]
trk["skill_vs_CLIPER_%"] = trk.apply(lambda r: M.skill(r.track_error_km, ref_t[r.lead_h]), axis=1)
ref_i = itn[itn.model == "SHIFOR"].set_index("lead_h")["mae_kt"]
itn["skill_vs_SHIFOR_%"] = itn.apply(lambda r: M.skill(r.mae_kt, ref_i[r.lead_h]), axis=1)
trk.to_csv(C.out("results") / "track_comparison.csv", index=False)
itn.to_csv(C.out("results") / "intensity_comparison.csv", index=False)

t_wide = trk.pivot(index="model", columns="lead_h", values="track_error_km").round(0)
t_wide.columns = [f"+{c} h (km)" for c in t_wide.columns]
i_wide = itn.pivot(index="model", columns="lead_h", values="mae_kt").round(1)
i_wide.columns = [f"+{c} h (kt)" for c in i_wide.columns]
display(t_wide.sort_values(t_wide.columns[-1])); display(i_wide.sort_values(i_wide.columns[-1]))
P.lead_lines(trk, "track_error_km", "Mean track error (km)", "Track error vs lead time (validation 2003-2004)",
             C.out("figures") / "track_error_by_model.png")
P.lead_lines(itn, "mae_kt", "Mean absolute error (kt)", "Intensity error vs lead time (validation 2003-2004)",
             C.out("figures") / "intensity_error_by_model.png")
P.comparison_table_image(t_wide.sort_values(t_wide.columns[-1]).reset_index(), "Track error (km), validation",
                         C.out("figures") / "table_track.png")
P.comparison_table_image(i_wide.sort_values(i_wide.columns[-1]).reset_index(), "Intensity MAE (kt), validation",
                         C.out("figures") / "table_intensity.png")

## Rapid intensification (classification) and peak intensity

In [ ]:
czoo = MD.classifier_zoo()
if FAST:
    czoo = {k: v for k, v in czoo.items() if k in ("Logistic Regression", "Random Forest", "XGBoost")}
ok_t, ok_v = tr.y_ri.notna().values, va.y_ri.notna().values
Xt, yt, gt = tr.loc[ok_t, FEATS], tr.loc[ok_t, "y_ri"].values.astype(int), tr.loc[ok_t, "year"].values
Xv, yv = va.loc[ok_v, FEATS], va.loc[ok_v, "y_ri"].values.astype(int)
ri_rows = [dict(model="Climatology (base rate)", brier=M.brier_binary(yv, np.full(len(yv), yt.mean())),
                roc_auc=0.5, pr_auc=yv.mean())]
for name, (make, grid) in czoo.items():
    params, _ = MD.tune(make, grid, Xt, yt, gt, kind="clf")
    m = make(**params).fit(Xt, yt)
    FITTED[(name, "y_ri")] = m
    p = m.predict_proba(Xv)[:, 1]
    ri_rows.append(dict(model=name, brier=M.brier_binary(yv, p),
                        roc_auc=roc_auc_score(yv, p) if len(set(yv)) > 1 else np.nan,
                        pr_auc=average_precision_score(yv, p) if len(set(yv)) > 1 else np.nan))
ri = pd.DataFrame(ri_rows).sort_values("brier"); ri.to_csv(C.out("results") / "ri_comparison.csv", index=False)
print(f"RI cases: train {yt.sum()} / {len(yt)}, val {yv.sum()} / {len(yv)}"); display(ri.round(4))

pk_rows = []
ok_t, ok_v = tr.y_peak_wind.notna().values, va.y_peak_wind.notna().values
yv_pk = va.loc[ok_v, "y_peak_wind"].values
pk_rows.append(dict(model="Persistence (peak = current)", mae_kt=M.mae(yv_pk, va.loc[ok_v, "wind"].values),
                    grade_accuracy=float((imd_grade(va.loc[ok_v, "wind"].values) == imd_grade(yv_pk)).mean())))
for name, (make, grid) in zoo.items():
    params, _ = MD.tune(make, grid, tr.loc[ok_t, FEATS], tr.loc[ok_t, "y_peak_wind"].values, tr.loc[ok_t, "year"].values)
    m = make(**params).fit(tr.loc[ok_t, FEATS], tr.loc[ok_t, "y_peak_wind"].values)
    FITTED[(name, "y_peak_wind")] = m
    p = np.maximum(m.predict(va.loc[ok_v, FEATS]), va.loc[ok_v, "wind"].values)   # peak can't be below current
    pk_rows.append(dict(model=name, mae_kt=M.mae(yv_pk, p), grade_accuracy=float((imd_grade(p) == imd_grade(yv_pk)).mean())))
pk = pd.DataFrame(pk_rows).sort_values("mae_kt"); pk.to_csv(C.out("results") / "peak_comparison.csv", index=False)
display(pk.round(3))

## Save the main models, error bank and uncertainty cone
`MAIN` is XGBoost (or the best available model if XGBoost is not installed).

In [ ]:
names = [n for n in zoo]
MAIN = "XGBoost" if "XGBoost" in names else min(names, key=lambda n: CV[(n, "dwind")])
for t in TARGET_TRACK + TARGET_INT + ["y_ri", "y_peak_wind"]:
    joblib.dump(FITTED[(MAIN, t)], C.out("models") / f"{t}.joblib")
json.dump({"main_model": MAIN, "params": {f"{k[0]}|{k[1]}": v for k, v in BEST.items()}},
          open(C.out("models") / "storm_models_info.json", "w"), indent=1, default=str)

# error bank: truth - prediction of the main track model, whole sequences, rows with all leads available
pr = PRED[MAIN]
full = va[[f"y_dlat_{L}" for L in C.LEADS_H] + [f"y_dlon_{L}" for L in C.LEADS_H]].notna().all(axis=1).values
err = np.column_stack([va[f"y_{c}_{L}"].values - np.asarray(pr[f"y_{c}_{L}"])
                       for c in ("dlat", "dlon") for L in C.LEADS_H])[full]
np.save(C.out("models") / "track_error_bank.npy", err)
cone = trk[trk.model == MAIN][["lead_h"]].copy()
cone["radius_km_p67"] = [np.nanpercentile(M.track_error_km(va[full], pr[full], L), 67) for L in C.LEADS_H]
cone.to_csv(C.out("models") / "cone_radius.csv", index=False)
print("main model:", MAIN, "| error bank rows:", len(err)); display(cone.round(0))

## Explain the main model (SHAP) and a readable surrogate tree

In [ ]:
for t in ("y_dwind_24", "y_dlat_24"):
    m = FITTED[(MAIN, t)]
    if MAIN == "XGBoost":
        import xgboost as xgb_mod
        contrib = m.get_booster().predict(xgb_mod.DMatrix(va[FEATS]), pred_contribs=True)[:, :-1]
        imp = pd.Series(np.abs(contrib).mean(axis=0), index=FEATS)
    else:
        imp = pd.Series(getattr(m, "feature_importances_", np.zeros(len(FEATS))), index=FEATS)
    imp.sort_values(ascending=False).to_csv(C.out("results") / f"importance_{t}.csv")
    P.importance_bar(imp, f"Top drivers of {t} ({MAIN})", C.out("figures") / f"importance_{t}.png")
ok = tr["y_dwind_24"].notna().values
sur = DecisionTreeRegressor(max_depth=3, min_samples_leaf=40).fit(
    tr.loc[ok, FEATS].fillna(tr[FEATS].median()), FITTED[(MAIN, "y_dwind_24")].predict(tr.loc[ok, FEATS]))
P.surrogate_tree(sur, FEATS, "How the model decides 24 h intensity change (simplified, depth 3)",
                 C.out("figures") / "surrogate_tree_dwind24.png")
print("figures saved to", C.out("figures"))